In [1]:
import pandas as pd
import numpy as np
import requests
import os

print("All libraries imported successfully!")


All libraries imported successfully!


In [2]:
# Create folder structure confirmation
folders = ['data/raw', 'data/processed', 'notebooks', 'sql', 'dashboard', 'reports']

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"✓ {folder}")

print("\nFolder structure ready!")

✓ data/raw
✓ data/processed
✓ notebooks
✓ sql
✓ dashboard
✓ reports

Folder structure ready!


In [3]:
import os
files = os.listdir("data/raw")
print("Files found:", files)


Files found: ['01_fund_master.csv', '02_nav_history.csv', '03_aum_by_fund_house.csv', '04_monthly_sip_inflows.csv', '05_category_inflows.csv', '06_industry_folio_count.csv', '07_scheme_performance.csv', '08_investor_transactions.csv', '09_portfolio_holdings.csv', '10_benchmark_indices.csv', 'Bluestock_MF_Capstone_Project.pdf']


In [4]:
# Load all 10 CSVs and print shape, dtypes, head
csv_files = [f for f in files if f.endswith('.csv')]

dataframes = {}

for file in csv_files:
    name = file.replace('.csv', '')
    df = pd.read_csv(f"data/raw/{file}")
    dataframes[name] = df
    print(f"\n{'='*50}")
    print(f"FILE: {file}")
    print(f"Shape: {df.shape}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nFirst 2 rows:\n{df.head(2)}")


FILE: 01_fund_master.csv
Shape: (40, 15)

Dtypes:
amfi_code               int64
fund_house             object
scheme_name            object
category               object
sub_category           object
plan                   object
launch_date            object
benchmark              object
expense_ratio_pct     float64
exit_load_pct         float64
min_sip_amount          int64
min_lumpsum_amount      int64
fund_manager           object
risk_category          object
sebi_category_code     object
dtype: object

First 2 rows:
   amfi_code       fund_house                                scheme_name  \
0     119551  SBI Mutual Fund  SBI Bluechip Fund - Regular Plan - Growth   
1     119552  SBI Mutual Fund   SBI Bluechip Fund - Direct Plan - Growth   

  category sub_category     plan launch_date      benchmark  \
0   Equity    Large Cap  Regular  2006-02-14  NIFTY 100 TRI   
1   Equity    Large Cap   Direct  2013-01-01  NIFTY 100 TRI   

   expense_ratio_pct  exit_load_pct  min_sip_amount

In [5]:
# Task 5 - Explore fund master
fm = dataframes['01_fund_master']

print("Unique Fund Houses:")
print(fm['fund_house'].unique())

print("\nUnique Categories:")
print(fm['category'].unique())

print("\nUnique Sub-Categories:")
print(fm['sub_category'].unique())

print("\nUnique Risk Grades:")
print(fm['risk_category'].unique())

Unique Fund Houses:
['SBI Mutual Fund' 'HDFC Mutual Fund' 'ICICI Prudential MF'
 'Nippon India MF' 'Kotak Mahindra MF' 'Axis Mutual Fund'
 'Aditya Birla Sun Life MF' 'UTI Mutual Fund' 'Mirae Asset MF'
 'DSP Mutual Fund']

Unique Categories:
['Equity' 'Debt']

Unique Sub-Categories:
['Large Cap' 'Small Cap' 'Gilt' 'Mid Cap' 'Short Duration' 'Value'
 'Liquid' 'Index/ETF' 'Flexi Cap' 'Index' 'Large & Mid Cap' 'ELSS']

Unique Risk Grades:
['Moderate' 'Very High' 'Low' 'High' 'Moderately High']


In [6]:
# Task 6 - Validate AMFI codes
nav = dataframes['02_nav_history']

master_codes = set(fm['amfi_code'])
nav_codes = set(nav['amfi_code'])

matching = master_codes & nav_codes
missing = master_codes - nav_codes

print(f"Total codes in fund_master: {len(master_codes)}")
print(f"Total unique codes in nav_history: {len(nav_codes)}")
print(f"Matching codes: {len(matching)}")
print(f"Missing codes: {len(missing)}")

if missing:
    print(f"Missing AMFI codes: {missing}")
else:
    print("All AMFI codes validated successfully!")

# Data quality summary
print("\n--- DATA QUALITY SUMMARY ---")
for name, df in dataframes.items():
    nulls = df.isnull().sum().sum()
    duplicates = df.duplicated().sum()
    print(f"{name}: {df.shape[0]} rows | {nulls} nulls | {duplicates} duplicates")

Total codes in fund_master: 40
Total unique codes in nav_history: 40
Matching codes: 40
Missing codes: 0
All AMFI codes validated successfully!

--- DATA QUALITY SUMMARY ---
01_fund_master: 40 rows | 0 nulls | 0 duplicates
02_nav_history: 46000 rows | 0 nulls | 0 duplicates
03_aum_by_fund_house: 90 rows | 0 nulls | 0 duplicates
04_monthly_sip_inflows: 48 rows | 12 nulls | 0 duplicates
05_category_inflows: 144 rows | 0 nulls | 0 duplicates
06_industry_folio_count: 21 rows | 0 nulls | 0 duplicates
07_scheme_performance: 40 rows | 0 nulls | 0 duplicates
08_investor_transactions: 32778 rows | 0 nulls | 0 duplicates
09_portfolio_holdings: 322 rows | 0 nulls | 0 duplicates
10_benchmark_indices: 8050 rows | 0 nulls | 0 duplicates


In [7]:
# Task 3 & 4 - Fetch live NAV from mfapi.in
schemes = {
    "HDFC_Top100": 125497,
    "SBI_Bluechip": 119551,
    "ICICI_Bluechip": 120503,
    "Nippon_LargeCap": 118632,
    "Axis_Bluechip": 119092,
    "Kotak_Bluechip": 120841
}

for name, code in schemes.items():
    response = requests.get(f"https://api.mfapi.in/mf/{code}")
    data = response.json()
    df = pd.DataFrame(data["data"])
    df.to_csv(f"data/raw/{name}_live.csv", index=False)
    latest_nav = data["data"][0]
    print(f"✓ {name} | Latest NAV: {latest_nav['nav']} | Date: {latest_nav['date']}")

print("\nAll live NAV data saved!")

✓ HDFC_Top100 | Latest NAV: 192.31950 | Date: 01-06-2026
✓ SBI_Bluechip | Latest NAV: 104.70250 | Date: 01-06-2026
✓ ICICI_Bluechip | Latest NAV: 103.09480 | Date: 01-06-2026
✓ Nippon_LargeCap | Latest NAV: 97.19440 | Date: 01-06-2026
✓ Axis_Bluechip | Latest NAV: 6156.75320 | Date: 01-06-2026
✓ Kotak_Bluechip | Latest NAV: 246.98140 | Date: 01-06-2026

All live NAV data saved!


In [9]:
import pandas as pd
import os

files = os.listdir("data/raw")
csv_files = [f for f in files if f.endswith(".csv")]

dataframes = {}

for file in csv_files:
    df = pd.read_csv(f"data/raw/{file}")
    dataframes[file] = df

    print("\n" + "="*50)
    print("FILE:", file)
    print("Shape:", df.shape)
    print(df.head(2))

fm = dataframes["01_fund_master.csv"]
nav = dataframes["02_nav_history.csv"]

master_codes = set(fm["amfi_code"])
nav_codes = set(nav["amfi_code"])

print("\nAMFI Validation")
print("Matching codes:", len(master_codes & nav_codes))
print("Missing codes:", len(master_codes - nav_codes))

print("\nData Quality Summary")
for name, df in dataframes.items():
    print(
        f"{name}: "
        f"{df.isnull().sum().sum()} nulls, "
        f"{df.duplicated().sum()} duplicates"
    )


FILE: 01_fund_master.csv
Shape: (40, 15)
   amfi_code       fund_house                                scheme_name  \
0     119551  SBI Mutual Fund  SBI Bluechip Fund - Regular Plan - Growth   
1     119552  SBI Mutual Fund   SBI Bluechip Fund - Direct Plan - Growth   

  category sub_category     plan launch_date      benchmark  \
0   Equity    Large Cap  Regular  2006-02-14  NIFTY 100 TRI   
1   Equity    Large Cap   Direct  2013-01-01  NIFTY 100 TRI   

   expense_ratio_pct  exit_load_pct  min_sip_amount  min_lumpsum_amount  \
0               1.54            1.0             500                1000   
1               0.66            1.0             500                1000   

    fund_manager risk_category sebi_category_code  
0  Sohini Andani      Moderate               EC01  
1  Sohini Andani      Moderate               EC01  

FILE: 02_nav_history.csv
Shape: (46000, 3)
   amfi_code        date      nav
0     119551  2022-01-03  54.3856
1     119551  2022-01-04  54.3474

FILE: 03_a

In [10]:
# live_nav_fetch.py

import requests
import pandas as pd

schemes = {
    "HDFC_Top100": 125497,
    "SBI_Bluechip": 119551,
    "ICICI_Bluechip": 120503,
    "Nippon_LargeCap": 118632,
    "Axis_Bluechip": 119092,
    "Kotak_Bluechip": 120841
}

for name, code in schemes.items():
    response = requests.get(f"https://api.mfapi.in/mf/{code}")

    if response.status_code == 200:
        data = response.json()

        df = pd.DataFrame(data["data"])
        df.to_csv(f"data/raw/{name}_live.csv", index=False)

        latest = data["data"][0]

        print(
            f"{name} | NAV: {latest['nav']} | Date: {latest['date']}"
        )

print("\nLive NAV fetch completed.")

HDFC_Top100 | NAV: 192.31950 | Date: 01-06-2026
SBI_Bluechip | NAV: 104.70250 | Date: 01-06-2026
ICICI_Bluechip | NAV: 103.09480 | Date: 01-06-2026
Nippon_LargeCap | NAV: 97.19440 | Date: 01-06-2026
Axis_Bluechip | NAV: 6156.75320 | Date: 01-06-2026
Kotak_Bluechip | NAV: 246.98140 | Date: 01-06-2026

Live NAV fetch completed.


In [11]:
import os

print(os.listdir())

['.ipynb_checkpoints', 'dashboard', 'data', 'data_ingestion.py', 'live_nav_fetch.py', 'notebooks', 'reports', 'sql', 'Untitled.ipynb']


In [12]:
summary = """
DAY 1 DATA QUALITY SUMMARY

Datasets Loaded: 10

AMFI Validation:
- Total fund codes: 40
- Matching codes: 40
- Missing codes: 0

Data Quality Findings:
- No duplicate records found
- No missing AMFI codes
- 12 null values found in monthly_sip_inflows
- All other datasets are clean

Live NAV fetched successfully for 6 schemes.
"""

with open("reports/day1_data_quality_summary.txt", "w") as f:
    f.write(summary)

print("Summary file created successfully!")

Summary file created successfully!


In [15]:
import os
import pandas as pd

folder_path = "../data/raw"

files = os.listdir(folder_path)
csv_files = [f for f in files if f.endswith(".csv")]

dataframes = {}

for file in csv_files:
    name = file.replace(".csv", "")
    df = pd.read_csv(os.path.join(folder_path, file))
    dataframes[name] = df

print("Loaded:", list(dataframes.keys()))

Loaded: ['01_fund_master', '02_nav_history', '03_aum_by_fund_house', '04_monthly_sip_inflows', '05_category_inflows', '06_industry_folio_count', '07_scheme_performance', '08_investor_transactions', '09_portfolio_holdings', '10_benchmark_indices', 'Axis_Bluechip_live', 'HDFC_Top100_live', 'ICICI_Bluechip_live', 'Kotak_Bluechip_live', 'Nippon_LargeCap_live', 'SBI_Bluechip_live']


In [16]:
dataframes.keys()

dict_keys(['01_fund_master', '02_nav_history', '03_aum_by_fund_house', '04_monthly_sip_inflows', '05_category_inflows', '06_industry_folio_count', '07_scheme_performance', '08_investor_transactions', '09_portfolio_holdings', '10_benchmark_indices', 'Axis_Bluechip_live', 'HDFC_Top100_live', 'ICICI_Bluechip_live', 'Kotak_Bluechip_live', 'Nippon_LargeCap_live', 'SBI_Bluechip_live'])